# FitResultExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `analysis`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/FitResultExamples.md`


Notebook source link: [FitResultExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/FitResultExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "FitResultExamples"
FAMILY = "analysis"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"FitResultExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"FitResultExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"FitResultExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"FitResultExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# FitResultExamples: fit GLM, inspect fit object, and plot diagnostics.
from nstat.compat.matlab import Analysis, FitResult

dt = 0.01
t = np.arange(0.0, 10.0, dt)
x1 = np.sin(2.0 * np.pi * 0.7 * t)
x2 = np.cos(2.0 * np.pi * 0.2 * t + 0.4)
X = np.column_stack([x1, x2])
eta = -1.9 + 0.8 * x1 - 0.45 * x2
lam = np.exp(eta)
y = rng.poisson(np.clip(lam * dt, 0.0, 0.9))

fit_native = Analysis.fitGLM(X=X, y=y, fitType="poisson", dt=dt)
fit = FitResult.fromStructure(fit_native.to_structure())
fit.parameter_labels = ["x1", "x2"]
fit.setFitResidual(Analysis.computeFitResidual(y=y, X=X, fit=fit, dt=dt))

lam_hat = fit.evalLambda(X)
aic = fit.getAIC()
bic = fit.getBIC()

fig, axes = plt.subplots(2, 1, figsize=(9.0, 6.0), sharex=False)
plt.sca(axes[0])
fit.plotCoeffs()
axes[0].set_title(f"{TOPIC}: coefficients")
axes[0].set_ylabel("weight")
axes[1].plot(t, lam, "k", linewidth=1.2, label="true")
axes[1].plot(t, lam_hat, "tab:blue", linewidth=1.0, label="fit")
axes[1].set_title("Lambda fit")
axes[1].set_xlabel("time [s]")
axes[1].set_ylabel("Hz")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

assert np.isfinite(aic) and np.isfinite(bic)
assert lam_hat.shape == lam.shape

CHECKPOINT_METRICS = {
    "aic": float(aic),
    "bic": float(bic),
    "lambda_rmse": float(np.sqrt(np.mean((lam_hat - lam) ** 2))),
}
CHECKPOINT_LIMITS = {
    "aic": (-1.0e6, 1.0e6),
    "bic": (-1.0e6, 1.0e6),
    "lambda_rmse": (0.0, 10.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
